# Loading Env

In [ ]:

from pinecone import Pinecone, ServerlessSpec
import os
from dotenv import load_dotenv
from datasets import load_dataset
import pandas as pd
import re 
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [3]:
load_dotenv(".env", override=True)

True

# Loading and reading data

In [16]:
pacovaldez_dataset = load_dataset("pacovaldez/stackoverflow-questions")

In [17]:
print(pacovaldez_dataset)
print(pacovaldez_dataset['train'][0])


DatasetDict({
    train: Dataset({
        features: ['title', 'body', 'label'],
        num_rows: 1572294
    })
    validation: Dataset({
        features: ['title', 'body', 'label'],
        num_rows: 785098
    })
    test: Dataset({
        features: ['title', 'body', 'label'],
        num_rows: 1570866
    })
})
{'title': 'Parsing json directly using input stream', 'body': '<p>I am doing a api call and in response i am getting json. So for that I want to parse it directly through input stream so that there would not be need of storing it in memory. \nFor this I am trying to use JSONReader but that i am unable use for api\'s less than 11.\nSo i dont know how to proceed with. I want it to be done from 2.0 version onwards. Even parsing through JsonReader is not working.\nI was thing of having GSON parser but i am not getting how to implement the same with Inputstream.</p>\n\n<p>EDIT:\nMy code for the same:</p>\n\n<pre><code>HttpClient client = new DefaultHttpClient();\nHttpGet httpG

In [18]:
df = pacovaldez_dataset['train'].select(range(5)).to_pandas()
df

,title,body,label
0,Parsing json directly using input stream,<p>I am doing a api call and in response i am ...,0
1,ZXing convert Bitmap to BinaryBitmap,"<p>I am using OpenCV and Zxing, and I'd like t...",0
2,Bundle ID in android,<p>What is meant by <strong>bundle ID</strong>...,0
3,"""sdkmanager: command not found"" after installi...",<p>I installed via <code>apt-get install andro...,0
4,Switch Case in c++,<p>How can I compare an array of char in c++ u...,0


In [19]:
small_pacovaldez_dataset = pacovaldez_dataset['train'].select(range(1000))

# Preprocessing text removing tag elements from body col

In [20]:
def clean_html_regex(text):
    clean = re.sub(r"<.*?>", " ", text)
    clean = re.sub(r"\s+", " ", clean)
    return clean.strip()

In [21]:
df["body_clean_tags"] = df["body"].apply(clean_html_regex)

In [22]:
print(df[["title", "body", "body_clean_tags"]].head(3))

                                      title  \
0  Parsing json directly using input stream   
1      ZXing convert Bitmap to BinaryBitmap   
2                      Bundle ID in android   

                                                body  \
0  <p>I am doing a api call and in response i am ...   
1  <p>I am using OpenCV and Zxing, and I'd like t...   
2  <p>What is meant by <strong>bundle ID</strong>...   

                                     body_clean_tags  
0  I am doing a api call and in response i am get...  
1  I am using OpenCV and Zxing, and I'd like to a...  
2  What is meant by bundle ID in android, What is...  


In [23]:
df["combined_text"] = df.apply(
    lambda row: f"Title: {row['title']} Body: {row['body_clean_tags']}", axis=1
)

In [24]:
pd.set_option("display.max_colwidth", None)
print(df[["title", "body_clean_tags", "combined_text"]].head(3))

                                      title  \
0  Parsing json directly using input stream   
1      ZXing convert Bitmap to BinaryBitmap   
2                      Bundle ID in android   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

# Database connection and setup index 

In [25]:
pc = Pinecone(api_key = os.environ.get("PINECONE_API_KEY"), environment = os.environ.get("PINECONE_ENV"))

In [26]:
pc.list_indexes()

[
    {
        "name": "text",
        "metric": "cosine",
        "host": "text-z3funzb.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 384,
        "deletion_protection": "disabled",
        "tags": null
    },
    {
        "name": "my-index",
        "metric": "cosine",
        "host": "my-index-z3funzb.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 384,
        "deletion_protection": "disabled",
        "tags": null
    },
    {
        "name": "pacovaldez"

In [27]:
index_name = "pacovaldez"
dimension = 384
metric = "cosine"

In [ ]:
pc.create_index(
    name = index_name, 
    dimension = dimension, 
    metric = metric, 
    spec = ServerlessSpec(
        cloud = "aws", 
        region = "us-east-1")
    )

In [29]:
index = pc.Index(name = index_name)

# Embedding the data

In [30]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [31]:
vec = model.encode("test", normalize_embeddings=True) 
len(vec), vec[:5]

(384,
 array([ 0.01157344,  0.0251362 , -0.03670187,  0.05932485, -0.00714908],
       dtype=float32))

In [32]:
sample_texts = df["combined_text"]
embs = model.encode(sample_texts, normalize_embeddings=True)  # shape: (3, 384)
[len(embs), len(embs[0])]

[5, 384]

In [33]:

BATCH = 200           
NAMESPACE = index_name
TEXT_TRUNC = 1000     

def iter_batches(frame, batch_size):
    for start in range(0, len(frame), batch_size):
        end = min(start + batch_size, len(frame))
        yield frame.iloc[start:end]

total = len(df)
pbar = tqdm(total=total, desc="Upserting to Pinecone")

for batch_df in iter_batches(df, BATCH):
    texts = batch_df["combined_text"].tolist()
    embs = model.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=False)

    vectors = []
    for (idx, row), vec in zip(batch_df.iterrows(), embs):
        vectors.append({
            "id": f"qa-{idx}", 
            "values": vec.tolist(),
            "metadata": {
                "title": row["title"],
                "text": row["combined_text"][:TEXT_TRUNC],
                "label": int(row["label"]),
                "source": "so-qa"
            }
        })

    index.upsert(vectors=vectors, namespace=NAMESPACE)
    pbar.update(len(batch_df))

pbar.close()
print("DONE")

Upserting to Pinecone: 100%|██████████| 5/5 [00:01<00:00,  4.51it/s]

DONE


# Building query

In [40]:
user_query = "ID in android"
qvec = model.encode(user_query, normalize_embeddings=True)

res = index.query(
    vector=qvec.tolist(),
    top_k=5,
    include_metadata=True,
    namespace=NAMESPACE
)
score_threshold = 0.6
for m in res["matches"]:
    # if score_threshold < round(m["score"], 2):
     print(f"{m['score']:.4f} → Title: {m['metadata']['title']}\n Text: {m['metadata']['text']}")


0.5532 → Title: Bundle ID in android
 Text: Title: Bundle ID in android Body: What is meant by bundle ID in android, What is its usage, And can two android apps have same bundle ID? if YES then why? and if NO then why
0.2062 → Title: "sdkmanager: command not found" after installing Android SDK
 Text: Title: "sdkmanager: command not found" after installing Android SDK Body: I installed via apt-get install android-sdk . However, doing a find / -name sdkmanager reveals there is no such binary anywhere on the system. On my Mac, the binary exists in $ANDROID_HOME/tools/bin . However, on the Ubuntu system (the system with the issue), the binary does not exist there: $ ls $ANDROID_HOME/tools/bin e2fsck fsck.ext4 mkfs.ext4 resize2fs screenshot2 tune2fs Where is the sdkmanager ? Edit: Not sure why the above didn't install sdkmanager , however, one solution I found was to install manually (instead of via apt-get) by downloading the Linux files at https://developer.android.com/studio/#downloads u